
# Demo Explanation: Customer Churn Preprocessing Pipeline

This file is for explaining the graded activity to the professor online. It uses the same main ideas from class: identify categorical data, separate nominal and ordinal features, handle missing values, encode categories, scale numerical columns, and train a simple classification model.


In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report



## Step 1: Create a small customer churn dataset

I created a simple customer churn dataset because the mini project asks for my own dataset. It contains nominal features, an ordinal feature, numerical features, missing values, and a target column.


In [2]:

churn_data = pd.DataFrame({
    'ContractType': ['Monthly', 'Yearly', 'Monthly', 'Two-Year', 'Monthly', 'Yearly',
                     'Monthly', 'Two-Year', 'Monthly', 'Yearly', 'Monthly', 'Two-Year'],
    'PaymentMethod': ['Credit Card', 'Bank Transfer', 'Credit Card', 'PayPal', np.nan, 'Bank Transfer',
                      'PayPal', 'Credit Card', 'Credit Card', 'Bank Transfer', 'PayPal', 'Credit Card'],
    'SatisfactionLevel': ['Low', 'High', 'Medium', 'High', 'Low', 'Medium',
                          np.nan, 'High', 'Low', 'Medium', 'Low', 'High'],
    'MonthlyCharges': [90, 60, 75, 55, 95, 70, 85, 50, np.nan, 65, 100, 52],
    'TenureMonths': [3, 24, 8, 36, 2, 18, 5, 40, 1, 20, 4, 45],
    'Churn': ['Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No']
})

churn_data


,ContractType,PaymentMethod,SatisfactionLevel,MonthlyCharges,TenureMonths,Churn
0,Monthly,Credit Card,Low,90.0,3,Yes
1,Yearly,Bank Transfer,High,60.0,24,No
2,Monthly,Credit Card,Medium,75.0,8,Yes
3,Two-Year,PayPal,High,55.0,36,No
4,Monthly,NaN,Low,95.0,2,Yes
5,Yearly,Bank Transfer,Medium,70.0,18,No
6,Monthly,PayPal,NaN,85.0,5,Yes
7,Two-Year,Credit Card,High,50.0,40,No
8,Monthly,Credit Card,Low,NaN,1,Yes
9,Yearly,Bank Transfer,Medium,65.0,20,No



## Step 2: Separate input features and target

`X` contains the columns used to make predictions. `y` contains the answer we want the model to predict, which is customer churn.


In [3]:

X = churn_data.drop('Churn', axis=1)
y = churn_data['Churn']

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(label_encoder.classes_)
print(y_encoded)


['No' 'Yes']
[1 0 1 0 1 0 1 0 1 0 1 0]



## Step 3: Identify feature types

- `ContractType` and `PaymentMethod` are nominal because they do not have a natural order.
- `SatisfactionLevel` is ordinal because Low, Medium, and High are ordered.
- `MonthlyCharges` and `TenureMonths` are numerical.


In [4]:

nominal_features = ['ContractType', 'PaymentMethod']
ordinal_features = ['SatisfactionLevel']
numeric_features = ['MonthlyCharges', 'TenureMonths']

satisfaction_order = [['Low', 'Medium', 'High']]



## Step 4: Build preprocessing transformers

For nominal features, I use `SimpleImputer` and `OneHotEncoder`. For the ordinal feature, I use `SimpleImputer` and `OrdinalEncoder`. For numerical features, I use `SimpleImputer` and `StandardScaler`.


In [5]:

nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=satisfaction_order))
])

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])



## Step 5: Combine preprocessing using ColumnTransformer

`ColumnTransformer` allows each group of columns to receive the correct preprocessing method.


In [6]:

preprocessor = ColumnTransformer(
    transformers=[
        ('nominal', nominal_transformer, nominal_features),
        ('ordinal', ordinal_transformer, ordinal_features),
        ('numeric', numeric_transformer, numeric_features)
    ]
)



## Step 6: Train the model pipeline

The pipeline applies preprocessing first, then trains a logistic regression model. This makes the workflow cleaner and reduces mistakes.


In [7]:

churn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.25, random_state=42, stratify=y_encoded
)

churn_pipeline.fit(X_train, y_train)
predictions = churn_pipeline.predict(X_test)

print('Accuracy:', accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions, target_names=label_encoder.classes_))


Accuracy: 1.0
              precision    recall  f1-score   support

          No       1.00      1.00      1.00         2
         Yes       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3




## Simple Speaking Script

I started by creating a customer churn dataset with categorical and numerical columns. Then I separated the target column, `Churn`, from the input features. After that, I identified which categorical columns were nominal and which one was ordinal.

For nominal features, I used one-hot encoding because there is no ranking between categories. For the ordinal feature, I used ordinal encoding because satisfaction has an order from Low to Medium to High. I also handled missing values using `SimpleImputer`, and I scaled the numerical columns using `StandardScaler`.

Finally, I placed all preprocessing steps and the logistic regression model inside one pipeline. This is useful because the same transformations are applied consistently during training and testing.
